# 04 — ModelOpt Q/DQ ONNX Export
Change `QUANT_DTYPE` to `'int8'`, `'fp8'`, or `'int4'` and run all.

In [3]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('../src').resolve()))

In [4]:
import numpy as np
from data import get_dataloader
from config import ExperimentConfig
from quant_modelopt import quantize_qdq

In [ ]:
QUANT_DTYPE    = "int8"           # 'int8' | 'fp8' | 'int4'
ONNX_IN        = "../onnx/resnet18.onnx"
ONNX_OUT       = f"../onnx/resnet18_{QUANT_DTYPE}_qdq.onnx"
CALIB_BATCHES  = 32

In [7]:
# Build calibration numpy array from dataloader
cfg = ExperimentConfig(device="cpu", batch_size=32)
loader = get_dataloader(cfg, split="val")

batches = []
for i, (images, _) in enumerate(loader):
    if i >= CALIB_BATCHES:
        break
    batches.append(images.numpy())

calib_data = np.concatenate(batches, axis=0)
print(f"Calibration data: {calib_data.shape}")

Calibration data: (1024, 3, 224, 224)


In [ ]:
quantize_qdq(ONNX_IN, calib_data, quant_dtype=QUANT_DTYPE, output_path=ONNX_OUT)
print(f"Saved → {ONNX_OUT}")

In [12]:
import onnx
ops = [n.op_type for n in onnx.load("onnx/resnet18_int8_qdq.onnx").graph.node]
print(f"Q: {ops.count('QuantizeLinear')}, DQ: {ops.count('DequantizeLinear')}")

Q: 41, DQ: 41
